# **frequency**

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import glob
from tqdm import tqdm

# 1. DCT Transformation Function
def compute_dct_from_array(img_array):
    img_float = np.float32(img_array) / 255.0
    dct_result = cv2.dct(img_float)
    dct_log = np.log(np.abs(dct_result) + 1e-8)
    dct_tensor = torch.tensor(dct_log, dtype=torch.float32).unsqueeze(0)
    dct_tensor = dct_tensor.repeat(3, 1, 1)
    return dct_tensor


# 2. Dataset Statistics Calculation
def compute_dataset_statistics(image_paths, img_size=(224, 224)):
    print("Calculating dataset statistics for global normalization...")
    all_dcts = []
    sample_paths = image_paths[:2000] if len(image_paths) > 2000 else image_paths

    for path in tqdm(sample_paths):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        img = cv2.resize(img, img_size)
        dct_tensor = compute_dct_from_array(img)
        all_dcts.append(dct_tensor)

    all_dcts = torch.stack(all_dcts)
    mean = all_dcts.mean(dim=[0, 2, 3])
    std = all_dcts.std(dim=[0, 2, 3])
    print(f"Global Mean: {mean}, Global Std: {std}")
    return mean, std

# 3. Custom Dataset Class
class DeepfakeDCTDataset(Dataset):
    def __init__(self, image_paths, labels, img_size=(224, 224), mean=None, std=None, spatial_transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.img_size = img_size
        self.spatial_transform = spatial_transform

        if mean is not None and std is not None:
            self.normalize = transforms.Normalize(mean=mean, std=std)
        else:
            self.normalize = nn.Identity()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Could not read image: {img_path}")

        img = cv2.resize(img, self.img_size)

        if self.spatial_transform:
            img_pil = Image.fromarray(img)
            img_pil = self.spatial_transform(img_pil)
            img = np.array(img_pil)

        dct_tensor = compute_dct_from_array(img)
        dct_tensor = self.normalize(dct_tensor)

        label_tensor = torch.tensor(label, dtype=torch.float32)
        return dct_tensor, label_tensor


# 4. Model Definition
def build_dct_model():
    model = models.efficientnet_b0(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, 1)
    return model


# 5. Training Loop
def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")
    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)

    best_val_loss = float('inf')
    patience = 5
    trigger_times = 0

    for epoch in range(epochs):
        # Training Phase
        model.train()
        train_loss = 0.0
        correct_train = 0
        total_train = 0

        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} - Train")
        for inputs, labels in loop:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs).squeeze(1)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            preds = torch.sigmoid(outputs) >= 0.5
            correct_train += (preds == labels).sum().item()
            total_train += labels.size(0)
            loop.set_postfix(loss=loss.item())

        train_acc = correct_train / total_train

        # Validation Phase
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs).squeeze(1)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                preds = torch.sigmoid(outputs) >= 0.5
                correct_val += (preds == labels).sum().item()
                total_val += labels.size(0)

        avg_val_loss = val_loss / len(val_loader)
        val_acc = correct_val / total_val

        print(f"Epoch {epoch+1} Summary: "
              f"Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            save_path = './models/best_dct_model_final.pth'
            torch.save(model.state_dict(), save_path)
            trigger_times = 0
            print(f"--> Model saved successfully to Drive: {save_path}")
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print("Early stopping triggered! The model is starting to overfit.")
                break

# 6. Main Execution Script

if __name__ == '__main__':
    import glob

    #Load training data
    train_real_images = glob.glob("./data/real_vs_fake/train/real/*.jpg")
    train_fake_images = glob.glob("./data/real_vs_fake/train/fake/*.jpg")

    # Load test data
    test_real_images = glob.glob("./data/real_vs_fake/test/real/*.jpg")
    test_fake_images = glob.glob("./data/real_vs_fake/test/fake/*.jpg")

    print(f"Train set: {len(train_real_images)} real, {len(train_fake_images)} fake")
    print(f"Test set: {len(test_real_images)} real, {len(test_fake_images)} fake")

    train_paths = train_real_images + train_fake_images
    train_labels = [0]*len(train_real_images) + [1]*len(train_fake_images)

    test_paths = test_real_images + test_fake_images
    test_labels = [0]*len(test_real_images) + [1]*len(test_fake_images)

    if len(train_paths) == 0 or len(test_paths) == 0:
        print("Error: No images found. Please check your dataset paths.")
    else:
        global_mean, global_std = compute_dataset_statistics(train_paths)


        spatial_transforms = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.15, contrast=0.15)
        ])

        train_dataset = DeepfakeDCTDataset(
            train_paths, train_labels,
            mean=global_mean, std=global_std,
            spatial_transform=spatial_transforms
        )
        val_dataset = DeepfakeDCTDataset(
            test_paths, test_labels,
            mean=global_mean, std=global_std,
            spatial_transform=None
        )

        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

        model = build_dct_model()
        train_model(model, train_loader, val_loader, epochs=35, lr=1e-3)

Train set: 50000 real, 50000 fake
Test set: 10000 real, 10000 fake
Calculating dataset statistics for global normalization...


100%|██████████| 2000/2000 [00:05<00:00, 343.45it/s]


Global Mean: tensor([-4.2779, -4.2779, -4.2779]), Global Std: tensor([1.4089, 1.4089, 1.4089])
Training on device: cuda


Epoch 1/35 - Train: 100%|██████████| 3125/3125 [14:41<00:00,  3.54it/s, loss=0.414]


Epoch 1 Summary: Train Loss: 0.6075, Train Acc: 0.6671 | Val Loss: 0.5821, Val Acc: 0.7104
--> Model saved successfully to Drive: /content/drive/MyDrive/best_dct_model_final.pth


...

Epoch 14 Summary: Train Loss: 0.2974, Train Acc: 0.8717 | Val Loss: 0.4839, Val Acc: 0.7903


Epoch 15/35 - Train: 100%|██████████| 3125/3125 [13:59<00:00,  3.72it/s, loss=0.199]


Epoch 15 Summary: Train Loss: 0.2794, Train Acc: 0.8798 | Val Loss: 0.4816, Val Acc: 0.8012


Epoch 16/35 - Train: 100%|██████████| 3125/3125 [13:58<00:00,  3.73it/s, loss=0.129]


Epoch 16 Summary: Train Loss: 0.2637, Train Acc: 0.8875 | Val Loss: 0.5232, Val Acc: 0.7887
Early stopping triggered! The model is starting to overfit.
